In [74]:
from langgraph.graph import StateGraph, START,END
from typing import TypedDict
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver

In [75]:
llm = ChatOpenAI()
load_dotenv()

True

In [76]:
class JokeState(TypedDict):
    topic:str
    joke:str
    explanation:str

In [77]:
def gen_joke(state:JokeState):
    prompt = f'generate the joke on {state['topic']}'
    response = llm.invoke(prompt).content
    return {'joke':response}

In [78]:
def gen_explanation(state:JokeState):
    prompt = f'generate the explanation on the topic {state['joke']}'
    response = llm.invoke(prompt).content
    return {'explanation':response}

In [79]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke',gen_joke)
graph.add_node('gen_explanation',gen_explanation)

graph.add_edge(START,'generate_joke')
graph.add_edge('generate_joke', 'gen_explanation')
graph.add_edge('gen_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)


In [80]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': 'Why did the pizza go to the doctor? Because it was feeling a little crusty!',
 'explanation': 'The joke "Why did the pizza go to the doctor? Because it was feeling a little crusty!" plays on the double meaning of the word "crusty". In one sense, "crusty" can refer to the crispy outer layer of a pizza crust. However, in another sense, "crusty" can also mean old, worn out, or in poor condition.\n\nTherefore, the joke suggests that the pizza went to the doctor because it was feeling "crusty" in the second sense - as if it were not in its best condition. The humor lies in the clever play on words and the unexpected twist of a pizza seeking medical attention. Overall, the joke is meant to be light-hearted and humorous, providing a silly explanation for why a pizza might need to see a doctor.'}

In [81]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the doctor? Because it was feeling a little crusty!', 'explanation': 'The joke "Why did the pizza go to the doctor? Because it was feeling a little crusty!" plays on the double meaning of the word "crusty". In one sense, "crusty" can refer to the crispy outer layer of a pizza crust. However, in another sense, "crusty" can also mean old, worn out, or in poor condition.\n\nTherefore, the joke suggests that the pizza went to the doctor because it was feeling "crusty" in the second sense - as if it were not in its best condition. The humor lies in the clever play on words and the unexpected twist of a pizza seeking medical attention. Overall, the joke is meant to be light-hearted and humorous, providing a silly explanation for why a pizza might need to see a doctor.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f18e327-08c2-650c-8002-aa5b7321f879'}}, metadata={'source':

In [82]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the doctor? Because it was feeling a little crusty!', 'explanation': 'The joke "Why did the pizza go to the doctor? Because it was feeling a little crusty!" plays on the double meaning of the word "crusty". In one sense, "crusty" can refer to the crispy outer layer of a pizza crust. However, in another sense, "crusty" can also mean old, worn out, or in poor condition.\n\nTherefore, the joke suggests that the pizza went to the doctor because it was feeling "crusty" in the second sense - as if it were not in its best condition. The humor lies in the clever play on words and the unexpected twist of a pizza seeking medical attention. Overall, the joke is meant to be light-hearted and humorous, providing a silly explanation for why a pizza might need to see a doctor.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f18e327-08c2-650c-8002-aa5b7321f879'}}, metadata={'source'

In [83]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'pasta'}, config=config2)

{'topic': 'pasta',
 'joke': 'Why did the pasta go to the party?\n\nBecause it heard it was going to be an al dente-tainment!',
 'explanation': 'Pasta went to the party because it heard from its friends that it was going to be a fun and entertaining event. The pasta was eager to attend and be a part of the festivities. Additionally, the play on words with "al dente-tainment" referencing pasta being cooked al dente added a humorous touch to the decision for why the pasta decided to go to the party. Ultimately, the pasta was looking forward to enjoying good company, delicious food, and a lively atmosphere at the party.'}

In [84]:
workflow.get_state(config2)

StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the pasta go to the party?\n\nBecause it heard it was going to be an al dente-tainment!', 'explanation': 'Pasta went to the party because it heard from its friends that it was going to be a fun and entertaining event. The pasta was eager to attend and be a part of the festivities. Additionally, the play on words with "al dente-tainment" referencing pasta being cooked al dente added a humorous touch to the decision for why the pasta decided to go to the party. Ultimately, the pasta was looking forward to enjoying good company, delicious food, and a lively atmosphere at the party.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f18e327-2589-6366-8002-cadff3621588'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-08-02T05:24:34.311241+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f18e327-1891-6ee2-8001-71a236f

In [85]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the pasta go to the party?\n\nBecause it heard it was going to be an al dente-tainment!', 'explanation': 'Pasta went to the party because it heard from its friends that it was going to be a fun and entertaining event. The pasta was eager to attend and be a part of the festivities. Additionally, the play on words with "al dente-tainment" referencing pasta being cooked al dente added a humorous touch to the decision for why the pasta decided to go to the party. Ultimately, the pasta was looking forward to enjoying good company, delicious food, and a lively atmosphere at the party.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f18e327-2589-6366-8002-cadff3621588'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-08-02T05:24:34.311241+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f18e327-1891-6ee2-8001-71a236

In [86]:
workflow.get_state({"configurable": {"thread_id": "2", "checkpoint_id": "1f06cc6e-7232-6cb1-8000-f71609e6cec5"}})

StateSnapshot(values={}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_id': '1f06cc6e-7232-6cb1-8000-f71609e6cec5'}}, metadata=None, created_at=None, parent_config=None, tasks=(), interrupts=())

In [91]:
workflow.invoke(None, {"configurable": {"thread_id": "2", "checkpoint_id": "1f18e327-090d-6b24-8000-59ca921108aa"}})

{'topic': 'pasta',
 'joke': 'Why did the pasta go to the party? Because it heard it was going to be a pasta-bilities-packed event!',
 'explanation': 'The pasta went to the party because it heard that it was going to be a pasta-bilities-packed event. This pun playfully suggests that the party will be full of possibilities and potential for the pasta to enjoy itself, socialize, and have a good time. It also highlights the fun and light-hearted nature of the event, making it seem like a must-attend occasion. Overall, the pasta decided to go to the party because it believed it would be a fun and exciting gathering with endless opportunities for enjoyment and connection.'}

In [92]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f18e326-ecac-6aac-8000-b7b9bf5b91dc"}})

{'topic': 'pizza',
 'joke': 'Why did the pizza maker go to therapy?\nBecause he kneaded to talk about his crust issues!',
 'explanation': 'The pizza maker went to therapy because he needed a safe space to address his underlying crust issues. Despite his outer appearance of perfection, deep down he was dealing with inner turmoil and self-doubt. Through therapy, he was able to explore his feelings, confront his fears, and work towards a healthier mindset. By kneading out his emotional struggles, he was able to create a more balanced and fulfilling life for himself.'}